In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
import os

In [4]:
llm=ChatGroq(model="qwen/qwen3.8-27b",api_key=os.getenv("GROQ_API_KEY"),max_tokens=50)

In [5]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explanation:str

In [6]:
def generate_joke(state:JokeState):
    prompt=f"Generate a small joke on the topic:{state['topic']}"
    response=llm.invoke(prompt).content
    
    return {
        "joke":response
    }

In [7]:
def generate_explanation(state:JokeState):
    prompt=f"Write an explanation in 1 line for joke: {state['joke']}"
    response=llm.invoke(prompt).content
    
    return {
        "explanation":response
    }

In [11]:
graph=StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

# checkpointer object of InMemorySaver class is created to save the state of the graph in memory. This allows for easy persistence and retrieval of the graph's state during execution.
checkpointer=InMemorySaver()

workflow=graph.compile(checkpointer=checkpointer)

In [13]:
config1={"configurable":{"thread_id":"1"}}
workflow.invoke({"topic":"pizza"},config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the doctor?\n\nBecause it wasn’t feeling very **crust-y**! 🍕',
 'explanation': 'The joke is a pun that plays on the homophone between "crusty" (a type of pizza crust) and "crusty" (a playful alteration of "crummy" or simply a state of being unwell), suggesting the pizza is'}

In [15]:
workflow.get_state(config1)
# access final state values using the checkpointer object

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor?\n\nBecause it wasn’t feeling very **crust-y**! 🍕', 'explanation': 'The joke is a pun that plays on the homophone between "crusty" (a type of pizza crust) and "crusty" (a playful alteration of "crummy" or simply a state of being unwell), suggesting the pizza is'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6ce3-a6f6-672a-8002-af29b3facba5'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-13T06:20:47.113189+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6ce3-a4da-65d0-8001-b8e673532895'}}, tasks=(), interrupts=())

In [16]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor?\n\nBecause it wasn’t feeling very **crust-y**! 🍕', 'explanation': 'The joke is a pun that plays on the homophone between "crusty" (a type of pizza crust) and "crusty" (a playful alteration of "crummy" or simply a state of being unwell), suggesting the pizza is'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6ce3-a6f6-672a-8002-af29b3facba5'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-13T06:20:47.113189+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6ce3-a4da-65d0-8001-b8e673532895'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor?\n\nBecause it wasn’t feeling very **crust-y**! 🍕'}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6ce3

In [17]:
config2={"configurable":{"thread_id":"2"}}
workflow.invoke({"topic":"football"},config=config2)

{'topic': 'football',
 'joke': 'Why did the soccer player stand up in the middle of the game?\n\nBecause he was **offside**!',
 'explanation': 'The humor relies on a pun where "offside," a soccer rule violation, is humorously reinterpreted as the physical action of "off" (moving away from) the "side" (sitting down).'}

In [18]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'football', 'joke': 'Why did the soccer player stand up in the middle of the game?\n\nBecause he was **offside**!', 'explanation': 'The humor relies on a pun where "offside," a soccer rule violation, is humorously reinterpreted as the physical action of "off" (moving away from) the "side" (sitting down).'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6cee-5269-63fd-8002-e2e1daade6a2'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-13T06:25:33.526937+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6cee-509f-682d-8001-1491c3e89ce9'}}, tasks=(), interrupts=())

In [19]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'football', 'joke': 'Why did the soccer player stand up in the middle of the game?\n\nBecause he was **offside**!', 'explanation': 'The humor relies on a pun where "offside," a soccer rule violation, is humorously reinterpreted as the physical action of "off" (moving away from) the "side" (sitting down).'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6cee-5269-63fd-8002-e2e1daade6a2'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-13T06:25:33.526937+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6cee-509f-682d-8001-1491c3e89ce9'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'football', 'joke': 'Why did the soccer player stand up in the middle of the game?\n\nBecause he was **offside**!'}, next=('generate_explanation',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6c

TIME TRAVEL

In [23]:
workflow.get_state(config={"configurable":{"thread_id":"2","checkpoint_id":"1f1c6cee-4ee4-6eb8-8000-9e414799f014"}})

StateSnapshot(values={'topic': 'football'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f1c6cee-4ee4-6eb8-8000-9e414799f014'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-10-13T06:25:33.158162+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6cee-4edf-6fdf-bfff-1c8d35f9f872'}}, tasks=(PregelTask(id='4d4d5fac-7088-0b98-6b27-f6a6cce0851c', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the soccer player stand up in the middle of the game?\n\nBecause he was **offside**!'}),), interrupts=())

In [24]:
workflow.invoke(None,config={"configurable":{"thread_id":"2","checkpoint_id":"1f1c6cee-4ee4-6eb8-8000-9e414799f014"}})

{'topic': 'football',
 'joke': 'Why did the football player break up with his girlfriend?\n\nBecause she said he was too **ball**-y.',
 'explanation': 'The joke is a pun that plays on the homophonic similarity between the slang term "bally" (meaning arrogant or tough) and the word "ball," which directly references the player\'s profession in football.'}

In [26]:
list(workflow.get_state_history({"configurable":{"thread_id":"2"}}))

[StateSnapshot(values={'topic': 'football', 'joke': 'Why did the football player break up with his girlfriend?\n\nBecause she said he was too **ball**-y.', 'explanation': 'The joke is a pun that plays on the homophonic similarity between the slang term "bally" (meaning arrogant or tough) and the word "ball," which directly references the player\'s profession in football.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6de4-f792-6c9f-8003-c79732ce5ddd'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-10-13T08:15:54.357673+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6de4-f5ad-61b1-8002-27cc78fab7c7'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'football', 'joke': 'Why did the football player break up with his girlfriend?\n\nBecause she said he was too **ball**-y.'}, next=('generate_explanation',), config={'configurable': {'thread_id': '2', '

Updating State

In [30]:
workflow.update_state(
    config={"configurable": {"thread_id": "2", "checkpoint_id": "1f1c6cee-4ee4-6eb8-8000-9e414799f014","checkpoint_ns":""}},
    values={"topic": "samosa"}
)

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1c6e4a-0e1d-6588-8001-f8f32673ccd9'}}

The checkpoint_ns (checkpoint namespace) is an internal identifier used to manage state in nested graphs or subgraphs . When update_state is called, the underlying Pregel engine validates the config and expects checkpoint_ns to be present 

In [34]:
list(workflow.get_state_history(config={"configurable":{"thread_id":"2"}}))

[StateSnapshot(values={'topic': 'football', 'joke': 'Why did the football player go to the doctor?\n\nBecause he had a bad knee.', 'explanation': 'The humor relies on a pun, playing on the double meaning of "bad knee" as both a physical injury to the leg joint and a derogatory slang term for a defective or useless person.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6e4f-59a8-6e77-8003-32847edbd96a'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-10-13T09:03:30.058610+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6e4f-57d1-6803-8002-bf12b6e7bc40'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'football', 'joke': 'Why did the football player go to the doctor?\n\nBecause he had a bad knee.'}, next=('generate_explanation',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6e4f-57d1-6803-8002-bf12b6e7bc40'

In [35]:
workflow.invoke(None,config={"configurable":{"thread_id":"2","checkpoint_id":"1f1c6e4a-0e1d-6588-8001-f8f32673ccd9"}})

{'topic': 'samosa',
 'joke': 'Why did the samosa go to the doctor?\n\nBecause it wasn’t feeling well in its **shell**! 🥟',
 'explanation': 'The joke is a pun on the phrase "feeling well in one\'s shell" (a state of contentment or protection), which is humorously applied to a samosa because its crispy outer crust is literally a "shell."'}

In [37]:
list(workflow.get_state_history({"configurable":{"thread_id":"2"}}))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa go to the doctor?\n\nBecause it wasn’t feeling well in its **shell**! 🥟', 'explanation': 'The joke is a pun on the phrase "feeling well in one\'s shell" (a state of contentment or protection), which is humorously applied to a samosa because its crispy outer crust is literally a "shell."'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6e52-f3f6-631d-8003-896b2f831089'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-10-13T09:05:06.768975+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6e52-f1f7-662a-8002-b113a191a018'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa go to the doctor?\n\nBecause it wasn’t feeling well in its **shell**! 🥟'}, next=('generate_explanation',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpo

So here after updating state in invoke function,we give checkpoint id of samosa the new topic to invoke it!